# VCD on POPE — reproduction and gated VCD (Colab runner)

Run the cells top to bottom. Use a GPU runtime (**Runtime → Change runtime type → L4 or A100**).
Results are written to Google Drive, so a disconnected session can be resumed by re-running the cells:
every run continues where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Repository, results on Drive, dependencies

In [ ]:
REPO_URL = "https://github.com/<YOUR_GITHUB_USER>/<YOUR_REPO>.git"   # <-- edit
DRIVE_RESULTS = "/content/drive/MyDrive/yzm5110_vcd_results"

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_RESULTS + "/runs", exist_ok=True)
if not os.path.exists("/content/vcd-pope"):
    !git clone $REPO_URL /content/vcd-pope
%cd /content/vcd-pope
!git pull --ff-only
!rm -rf results/runs && mkdir -p results && ln -sfn $DRIVE_RESULTS/runs results/runs
!git rev-parse HEAD

In [ ]:
!pip install -q -r requirements.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 2. Data (POPE question files + ~1000 COCO val2014 images) and unit tests

In [ ]:
!python scripts/download_data.py
!python -m pytest -q tests

## 3. Smoke test — 100 questions each; read the per-question latency to estimate total GPU time
Expected: VCD ≈ 2× the latency of regular decoding; the lossless gate should be faster than VCD.

In [ ]:
!bash scripts/experiments.sh smoke
!python scripts/make_tables.py && cat results/tables/main.md results/tables/agreement.md

## 4. Phase 1 — reproduction (COCO POPE, 3 splits)
Long. If the session stops, re-run this cell; finished questions are skipped.
To split the work over several sessions you can run one split at a time, e.g. `!SEEDS="1" bash scripts/experiments.sh repro`.

In [ ]:
!bash scripts/experiments.sh repro

## 5. Phase 2a — choose the gate threshold on the dev split (A-OKVQA POPE)

In [ ]:
!bash scripts/experiments.sh dev

## 6. Phase 2b — improvement on COCO (set TAU and SKIP from the output above)

In [ ]:
TAU = "0.9"     # <-- from pick_tau.py
SKIP = "0.5"    # <-- from pick_tau.py
!TAU=$TAU SKIP=$SKIP bash scripts/experiments.sh test

## 7. Tables and a zip of all outputs (commit these to the repository)

In [ ]:
!python scripts/make_tables.py
!cat results/tables/main.md
!cd results && zip -qr /content/drive/MyDrive/yzm5110_vcd_results/results_$(date +%Y%m%d_%H%M).zip runs tables
print("zip saved to Drive")